In [23]:
# =====================================================================
# 🎬 VJ MIX GENERATOR v3 — telão sem áudio (Google Colab)
# Cole numa célula do Colab. Usa video_files e OUTPUT se já existirem;
# senão lê de VIDEO_DIR / grava em OUTPUT (ajuste abaixo).
#
# NOVIDADES v3
#  • Estrutura do set: breakdown / build / DROP (por análise do áudio,
#    por timeline digitada, ou ciclo automático)
#  • Efeitos: feedback (rastro), caleidoscópio, neon por contorno,
#    gradient map (paleta que gira), onda, punch-in no beat, pixelate +
#    gagueira antes do drop, flash suave no drop, título em máscara,
#    camadas geradas (túnel, lasers, grid), scanlines
#  • Cortes alinhados na grade de batidas (do áudio ou do BPM)
# =====================================================================
import os, glob, math, re, zlib, random, shutil, subprocess, time
from bisect import bisect_right
from collections import deque
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

VIDEO_DIR = "/content/drive/MyDrive/VJ/videos"
AUDIO_DIR = "/content/drive/MyDrive/VJ/audios"   # só para ANÁLISE (o vídeo sai mudo)
if "video_files" not in globals():
    video_files = sorted(sum([glob.glob(f"{VIDEO_DIR}/*{e}") for e in
                              (".mp4", ".mov", ".mkv", ".avi", ".webm")], []))
if "audio_files" not in globals():
    audio_files = sorted(sum([glob.glob(f"{AUDIO_DIR}/*{e}") for e in
                              (".mp3", ".wav", ".flac", ".m4a", ".ogg")], []))
if "OUTPUT" not in globals():
    OUTPUT = "/content/drive/MyDrive/VJ/saida"
os.makedirs(OUTPUT, exist_ok=True)

BLOCO_SEG = 300            # renderiza em blocos de 5 min (retoma se a sessão cair)
TITULO_INTERVALO = 60      # título aparece a cada N segundos...
TITULO_VISIVEL = 5         # ...e fica N segundos na tela
FLASH_DROP = 80            # brilho máx. do flash no drop (médio, nunca branco total)

# ---------------------------------------------------------------------
# 1. PERFIS
# ---------------------------------------------------------------------
PERFIS_GENERO = {
    "Melodic Techno":    dict(corte=(3.5, 8.0),  ovl=0.45, rgb=0.18, vhs=0.08, fade=0.85, contraste=1.25, brilho=-15),
    "Progressive House": dict(corte=(4.5, 9.5),  ovl=0.55, rgb=0.10, vhs=0.10, fade=0.90, contraste=1.10, brilho=-5),
    "Melodic House":     dict(corte=(4.0, 8.5),  ovl=0.50, rgb=0.08, vhs=0.12, fade=0.85, contraste=1.05, brilho=0),
    "Deep House":        dict(corte=(5.0, 10.0), ovl=0.40, rgb=0.05, vhs=0.20, fade=0.90, contraste=1.05, brilho=-5),
    "Techno":            dict(corte=(1.5, 4.0),  ovl=0.30, rgb=0.28, vhs=0.25, fade=0.35, contraste=1.35, brilho=-20),
    "Tech House":        dict(corte=(2.0, 4.5),  ovl=0.25, rgb=0.20, vhs=0.15, fade=0.50, contraste=1.15, brilho=0),
    "Minimal":           dict(corte=(3.0, 6.5),  ovl=0.35, rgb=0.15, vhs=0.15, fade=0.60, contraste=1.20, brilho=-10),
    "Organic House":     dict(corte=(5.0, 11.0), ovl=0.60, rgb=0.04, vhs=0.15, fade=0.95, contraste=1.00, brilho=0),
    "Afro House":        dict(corte=(3.0, 6.0),  ovl=0.40, rgb=0.12, vhs=0.10, fade=0.70, contraste=1.15, brilho=0),
}
# fx = multiplicador da chance de cada efeito; hue = cor-base (OpenCV 0-179); hue_vel = giro por segundo
ESTETICAS = {
    "Underground / Analog":   dict(vhs=1.8, rgb=0.8, pb=0.15, hue=12,  hue_vel=0.10,
                                   fx=dict(fb=1.0, gmap=0.7, wave=0.6, kaleido=0.5, neon=0.4, ger=0.6, scan=2.0)),
    "Experimental / Glitch":  dict(vhs=1.4, rgb=2.2, pb=0.05, hue=150, hue_vel=1.00,
                                   fx=dict(fb=1.3, gmap=1.0, wave=1.5, kaleido=1.4, neon=1.0, ger=1.0, scan=0.6)),
    "TV / Found Footage":     dict(vhs=2.5, rgb=1.2, pb=0.25, hue=85,  hue_vel=0.10,
                                   fx=dict(fb=0.6, gmap=0.4, wave=0.5, kaleido=0.3, neon=0.3, ger=0.3, scan=2.5)),
    "Psychedelic / Abstract": dict(vhs=0.7, rgb=1.8, pb=0.00, hue=140, hue_vel=2.00,
                                   fx=dict(fb=1.5, gmap=1.6, wave=1.8, kaleido=1.8, neon=0.6, ger=1.0, scan=0.0)),
    "Dark / Industrial":      dict(vhs=1.2, rgb=1.4, pb=0.45, hue=105, hue_vel=0.15,
                                   fx=dict(fb=0.9, gmap=0.6, wave=0.6, kaleido=0.9, neon=1.8, ger=1.2, scan=1.2)),
}
PALETAS = {"Auto (pela estética)": None, "Ciano / azul": 100, "Roxo": 130, "Magenta": 150,
           "Vermelho": 0, "Laranja": 12, "Verde": 65}
# chance-base de cada efeito por FASE do set
POOL = {
    "breakdown": dict(fb=.45, gmap=.50, wave=.35, kaleido=.15, neon=.10, ger=.25, scan=.10),
    "build":     dict(fb=.25, gmap=.40, wave=.20, kaleido=.25, neon=.20, ger=.30, scan=.10),
    "drop":      dict(fb=.20, gmap=.30, wave=.15, kaleido=.40, neon=.35, ger=.30, scan=.10),
}
RESOLUCOES = {
    "1280x720 (16:9) - rápido":      (1280, 720),
    "1920x1080 (16:9) - Full HD":    (1920, 1080),
    "2560x1080 (21:9) - ultrawide":  (2560, 1080),
    "3840x2160 (16:9) - 4K (lento)": (3840, 2160),
    "1080x1920 (9:16) - vertical":   (1080, 1920),
}

# ---------------------------------------------------------------------
# 2. EFEITOS (todos devolvem um frame NOVO; nada modifica o frame de entrada)
# ---------------------------------------------------------------------
def cobrir(frame, W, H):
    """Redimensiona SEM distorcer: preenche a tela e corta o excesso."""
    h, w = frame.shape[:2]
    s = max(W / w, H / h)
    nw, nh = int(math.ceil(w * s)), int(math.ceil(h * s))
    interp = cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR
    f = cv2.resize(frame, (nw, nh), interpolation=interp)
    x, y = (nw - W) // 2, (nh - H) // 2
    return f[y:y + H, x:x + W]

def grade(frame, contraste, brilho, pb=False):
    if pb:
        frame = cv2.cvtColor(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
    x = np.arange(256, dtype=np.float32)
    lut = np.clip((x - 128) * contraste + 128 + brilho, 0, 255).astype(np.uint8)
    return cv2.LUT(frame, lut)

def rgb_split(f, d):
    out = f.copy()
    out[:, :, 0] = np.roll(f[:, :, 0], d, axis=1)
    out[:, :, 2] = np.roll(f[:, :, 2], -d, axis=1)
    return out

def vhs_glitch(f):
    h, w = f.shape[:2]
    out = f.copy()
    for _ in range(random.randint(2, 5)):
        y = random.randint(0, h - 2)
        bh = random.randint(max(2, h // 60), max(3, h // 12))
        dx = random.randint(-w // 20, w // 20)
        out[y:y + bh] = np.roll(out[y:y + bh], dx, axis=1)
    n = np.random.randint(0, 20, (max(1, h // 4), max(1, w // 4)), dtype=np.uint8)
    n = cv2.resize(n, (w, h), interpolation=cv2.INTER_NEAREST)
    return cv2.add(out, cv2.merge([n, n, n]))

def blend(a, b, op, modo):
    """Mistura de camadas. 'screen' e 'lighten' mantêm o fundo escuro (bom pra LED)."""
    if modo == "screen":
        s = cv2.bitwise_not(cv2.multiply(cv2.bitwise_not(a), cv2.bitwise_not(b), scale=1 / 255.0))
    else:
        s = cv2.max(a, b)
    return cv2.addWeighted(a, 1 - op, s, op, 0)

def zoom_rot(f, s, ang=0.0):
    """Zoom (s>1 aproxima) e rotação em torno do centro."""
    if s < 1.002 and abs(ang) < 0.05:
        return f
    h, w = f.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, s)
    return cv2.warpAffine(f, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

def onda(f, t, amp, freq=3.0):
    """Ondulação líquida (usa só senos 1D + broadcast: barato)."""
    h, w = f.shape[:2]
    ys = np.arange(h, dtype=np.float32)
    xs = np.arange(w, dtype=np.float32)
    ox = (amp * np.sin(ys * (2 * math.pi * freq / h) + t * 2.0)).astype(np.float32)
    oy = (amp * 0.6 * np.sin(xs * (2 * math.pi * freq * 1.3 / w) + t * 1.6)).astype(np.float32)
    mx = xs[None, :] + ox[:, None]
    my = ys[:, None] + oy[None, :]
    return cv2.remap(f, mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

def mapas_kaleido(W, H, N):
    y, x = np.indices((H, W), dtype=np.float32)
    dx, dy = x - W / 2, y - H / 2
    r = np.sqrt(dx * dx + dy * dy)
    th = np.arctan2(dy, dx)
    seg = 2 * np.pi / N
    t = np.mod(th, seg)
    t = np.minimum(t, seg - t) - seg / 4          # fatia espelhada, centrada na horizontal
    esc = 0.55
    mx = (W / 2 + r * esc * np.cos(t)).astype(np.float32)
    my = (H / 2 + r * esc * np.sin(t)).astype(np.float32)
    return mx, my

def kaleido(f, mapas, ang):
    f = zoom_rot(f, 1.0, ang % 360)
    return cv2.remap(f, mapas[0], mapas[1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

def pixelar(f, bs):
    h, w = f.shape[:2]
    small = cv2.resize(f, (max(1, w // bs), max(1, h // bs)), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)

def paleta_lut(h):
    """3 paradas (escuro -> médio -> claro) na matiz h, interpoladas em 256 níveis."""
    hsv = np.array([[[h % 180, 255, 30]], [[(h + 22) % 180, 235, 150]], [[(h + 55) % 180, 170, 255]]], np.uint8)
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[:, 0, :].astype(np.float32)
    xs = np.arange(256)
    lut = np.stack([np.interp(xs, [0, 128, 255], bgr[:, c]) for c in range(3)], axis=1).astype(np.uint8)
    return lut, bgr

def gradient_map(f, lut, mix):
    gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
    return cv2.addWeighted(f, 1 - mix, lut[gray], mix, 0)

def neon(f, cor):
    """Contornos brilhantes sobre fundo escurecido (look wireframe)."""
    h, w = f.shape[:2]
    small = cv2.resize(f, (w // 2, h // 2), interpolation=cv2.INTER_AREA)
    g = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (3, 3), 0)
    e = cv2.dilate(cv2.Canny(g, 50, 130), np.ones((2, 2), np.uint8))
    lay = (e[..., None].astype(np.float32) * (np.asarray(cor, np.float32) / 255.0)).astype(np.uint8)
    lay = cv2.add(lay, cv2.GaussianBlur(lay, (0, 0), max(1.0, w / 500)))
    lay = cv2.resize(lay, (w, h), interpolation=cv2.INTER_LINEAR)
    return cv2.add(cv2.convertScaleAbs(f, alpha=0.3), lay)

def scanlines(f):
    out = f.copy()
    out[1::3] = cv2.convertScaleAbs(out[1::3], alpha=0.72)
    return out

def camada_gerada(tipo, t, W, H, cor, pulso):
    """Camadas procedurais (túnel / lasers / grid), desenhadas em meia resolução + glow."""
    w2, h2 = W // 2, H // 2
    c = np.zeros((h2, w2, 3), np.uint8)
    cor = tuple(int(x) for x in cor)
    if tipo == "tunel":
        cx, cy, N = w2 // 2, h2 // 2, 9
        frac = (t * 0.5) % 1.0
        for k in range(N):
            s = (k + frac) / N
            s2 = s ** 2.3
            hw, hh = int(s2 * w2 * 0.75), int(s2 * h2 * 0.75)
            if hw < 2 or hh < 2:
                continue
            br = min(1.0, 0.25 + s)
            col = tuple(int(x * br) for x in cor)
            cv2.rectangle(c, (cx - hw, cy - hh), (cx + hw, cy + hh), col, max(1, int(1 + pulso * 3)))
    elif tipo == "lasers":
        L = int(math.hypot(w2, h2))
        for k in range(10):
            a = math.sin(t * 0.55 + k * 0.9) * 0.9 + (k - 5) * 0.12
            cv2.line(c, (w2 // 2, 0), (int(w2 / 2 + L * math.sin(a)), int(L * math.cos(a))), cor, 1 + int(pulso * 2))
            b = math.sin(t * 0.4 + k * 1.3 + 2.0) * 0.9 + (k - 5) * 0.10
            cv2.line(c, (w2 // 2, h2), (int(w2 / 2 + L * math.sin(b)), int(h2 - L * math.cos(b))), cor, 1)
    else:  # grade em perspectiva
        hor, cx = int(h2 * 0.52), w2 // 2
        for i in range(-14, 15):
            cv2.line(c, (cx + int(i * w2 * 0.008), hor), (cx + int(i * w2 * 0.11), h2), cor, 1)
        frac = (t * 0.7) % 1.0
        for k in range(12):
            y = hor + int(((k + frac) / 12) ** 2.0 * (h2 - hor))
            cv2.line(c, (0, y), (w2, y), cor, 1)
    c = cv2.add(c, cv2.GaussianBlur(c, (0, 0), max(1.5, w2 / 250)))
    return cv2.resize(c, (W, H), interpolation=cv2.INTER_LINEAR)

def desenhar_titulo(f, texto, alpha, W, H):
    over = f.copy()
    cv2.putText(over, texto, (int(W * 0.06), int(H * 0.92)), cv2.FONT_HERSHEY_DUPLEX,
                H / 720 * 1.1, (255, 255, 255), max(1, H // 360), cv2.LINE_AA)
    return cv2.addWeighted(over, alpha, f, 1 - alpha, 0)

class TituloMask:
    """Título como 'janela': o vídeo aparece dentro das letras, o resto escurece."""
    def __init__(self, texto, W, H):
        fonte = cv2.FONT_HERSHEY_DUPLEX
        (tw, th), _ = cv2.getTextSize(texto, fonte, 1.0, 2)
        esc = min((W * 0.78) / max(tw, 1), (H * 0.28) / max(th, 1))
        grossura = max(2, int(esc * 4))
        (tw, th), _ = cv2.getTextSize(texto, fonte, esc, grossura)
        m = np.zeros((H, W), np.uint8)
        cv2.putText(m, texto, ((W - tw) // 2, (H + th) // 2), fonte, esc, 255, grossura, cv2.LINE_AA)
        self.m3 = cv2.merge([m, m, m])

    def aplicar(self, f, a):
        masked = cv2.add(cv2.bitwise_and(f, self.m3), cv2.convertScaleAbs(f, alpha=0.10))
        return cv2.addWeighted(f, 1 - a, masked, a, 0)

# ---------------------------------------------------------------------
# 3. LEITURA DE CLIPES (respeita o FPS da origem, sem acelerar/frear)
# ---------------------------------------------------------------------
class Clip:
    def __init__(self, path, W, H, fps_out, n_frames):
        self.cap = cv2.VideoCapture(path)
        self.W, self.H = W, H
        src = self.cap.get(cv2.CAP_PROP_FPS)
        if not src or src < 1 or src > 240:
            src = fps_out
        self.step, self.acc, self.last = src / fps_out, 0.0, None
        total = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        need = int(n_frames * self.step) + 2
        if total > need:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, random.randint(0, total - need))

    def read(self):
        self.acc += self.step
        n = int(self.acc)
        self.acc -= n
        if n >= 1:
            for _ in range(n - 1):
                self.cap.grab()
            ok, f = self.cap.read()
            if not ok:                       # acabou o arquivo: volta ao início (loop)
                self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                ok, f = self.cap.read()
            if ok:
                self.last = cobrir(f, self.W, self.H)
        if self.last is None:
            self.last = np.zeros((self.H, self.W, 3), np.uint8)
        return self.last                     # nunca modifique este array in-place

    def close(self):
        self.cap.release()

def validar_videos(lista):
    ok = []
    for p in lista:
        cap = cv2.VideoCapture(p)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        good, _ = cap.read()
        cap.release()
        if good and n > 0:
            ok.append(p)
        else:
            print(f"⚠️ ignorando (não abriu): {os.path.basename(p)}")
    return ok

# ---------------------------------------------------------------------
# 4. ESTRUTURA DO SET (breakdown / build / drop) + GRADE DE BATIDAS
# ---------------------------------------------------------------------
def mmss(s):
    return f"{int(s // 60)}:{int(s % 60):02d}"

class Estrutura:
    def __init__(self, segs, tv, vv, beats, origem):
        self.segs = [(float(a), float(b), n) for a, b, n in segs]
        self.starts = [s[0] for s in self.segs]
        self.tv, self.vv = np.asarray(tv, float), np.asarray(vv, float)
        self.quedas = [a for a, b, n in self.segs if n == "drop" and a > 0.5]
        self.beats, self.origem = beats, origem

    def fase(self, t):
        return self.segs[max(bisect_right(self.starts, t) - 1, 0)][2]

    def intens(self, t):
        return float(np.interp(t, self.tv, self.vv))

    def batida(self, t):
        """(posição 0..1 dentro da batida, força) ou None. Força 1.0 a cada 4 batidas (aprox.)."""
        b = self.beats
        if b is None or len(b) < 2:
            return None
        k = int(np.searchsorted(b, t, side="right")) - 1
        if k < 0 or k >= len(b) - 1:
            return None
        return (t - b[k]) / (b[k + 1] - b[k]), (1.0 if k % 4 == 0 else 0.4)

    def resumo(self):
        nomes = {"breakdown": "breakdown", "build": "build", "drop": "DROP"}
        l = [f"{mmss(a)}-{mmss(b)} {nomes[n]}" for a, b, n in self.segs[:14]]
        extra = f" (+{len(self.segs) - 14} trechos)" if len(self.segs) > 14 else ""
        return f"🧭 Estrutura [{self.origem}]: " + " | ".join(l) + extra + f" | {len(self.quedas)} drops"

def curva_de_segmentos(segs):
    niveis = {"breakdown": (0.25, 0.25), "build": (0.40, 0.85), "drop": (0.92, 0.92)}
    tv, vv = [], []
    for a, b, n in segs:
        va, vb = niveis[n]
        tv += [a + 1e-3, b - 1e-3]
        vv += [va, vb]
    return np.array(tv), np.array(vv)

def parse_tempo(s):
    """'4:30' = 4min30s | '1:02:00' = 1h02 | '4' = 4 minutos."""
    p = s.strip().replace(",", ".").split(":")
    if len(p) == 1:
        return float(p[0]) * 60
    seg = 0.0
    for x in p:
        seg = seg * 60 + float(x)
    return seg

def estrutura_timeline(texto, beats):
    nomes = {"breakdown": "breakdown", "break": "breakdown", "calmo": "breakdown",
             "build": "build", "subida": "build", "drop": "drop", "queda": "drop"}
    segs = []
    for m in re.finditer(r"([\d:.,]+)\s*[-–]\s*([\d:.,]+)\s*([a-zçã]+)", texto.lower()):
        n = nomes.get(m.group(3))
        a, b = parse_tempo(m.group(1)), parse_tempo(m.group(2))
        if n and b > a:
            segs.append((a, b, n))
    if not segs:
        return None
    segs.sort()
    final, t = [], 0.0
    for a, b, n in segs:
        a = max(a, t)
        if b <= a:
            continue
        if a > t + 0.5:
            final.append((t, a, "breakdown"))
        final.append((a, b, n))
        t = b
    return Estrutura(final, *curva_de_segmentos(final), beats, "timeline")

def estrutura_auto(dur, ciclo_min, beats):
    ciclo = min(max(ciclo_min, 1) * 60.0, dur)
    partes = [("breakdown", .15), ("build", .25), ("drop", .45), ("breakdown", .15)]
    segs, t = [], 0.0
    while t < dur:
        for n, fr in partes:
            segs.append((t, t + ciclo * fr, n))
            t += ciclo * fr
            if t >= dur:
                break
    return Estrutura(segs, *curva_de_segmentos(segs), beats, f"ciclo {ciclo / 60:.1f} min")

def _fundir(segs, min_len):
    segs = [list(s) for s in segs]
    while True:
        out = []
        for s in segs:
            if out and out[-1][2] == s[2]:
                out[-1][1] = s[1]
            else:
                out.append(list(s))
        segs = out
        curtos = [k for k, s in enumerate(segs) if s[1] - s[0] < min_len]
        if not curtos or len(segs) == 1:
            return segs
        k = min(curtos, key=lambda k: segs[k][1] - segs[k][0])
        if k > 0:
            segs[k - 1][1] = segs[k][1]
        else:
            segs[1][0] = segs[0][0]
        segs.pop(k)

def fases_de_curva(tv, vv, min_len=12.0):
    lento = np.interp(tv - 8.0, tv, vv)
    slope = vv - lento
    nomes = []
    for v, s in zip(vv, slope):
        if v >= 0.62:
            n = "drop"
        elif v < 0.35:
            n = "breakdown"
        elif s > 0.05:
            n = "build"
        elif s < -0.05:
            n = "breakdown"
        else:
            n = "build" if v >= 0.5 else "breakdown"
        nomes.append(n)
    segs, ini = [], 0
    for k in range(1, len(nomes) + 1):
        if k == len(nomes) or nomes[k] != nomes[ini]:
            segs.append((tv[ini], tv[k] if k < len(tv) else tv[-1] + 0.25, nomes[ini]))
            ini = k
    return _fundir(segs, min_len)

def normalizar_batidas(b):
    b = np.asarray(b, float)
    if len(b) < 16:
        return None
    per = float(np.median(np.diff(b)))
    while per > 0.70:                                   # tracker pegou meio-tempo
        b = np.sort(np.concatenate([b, (b[:-1] + b[1:]) / 2]))
        per = float(np.median(np.diff(b)))
    while per < 0.34:                                   # tracker pegou tempo dobrado
        b = b[::2]
        per = float(np.median(np.diff(b)))
    return b

def estrutura_audio(path, dur):
    import librosa
    print("🎧 Analisando o áudio (energia + batidas); pode levar de 1 a 3 min...")
    y, sr = librosa.load(path, sr=11025, mono=True, duration=dur)
    hop = 256
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    tt = librosa.times_like(rms, sr=sr, hop_length=hop)
    tv = np.arange(0, max(min(dur, float(tt[-1])), 1.0), 0.25)
    e = np.interp(tv, tt, rms)
    k = 12                                              # média de 3 s
    e = np.convolve(e, np.ones(k) / k, mode="same")
    lo, hi = np.percentile(e, 5), np.percentile(e, 95)
    vv = np.clip((e - lo) / (hi - lo + 1e-9), 0, 1)
    segs = fases_de_curva(tv, vv)
    beats = None
    try:
        _, b = librosa.beat.beat_track(y=y, sr=sr, hop_length=hop, units="time")
        beats = normalizar_batidas(b)
    except Exception as ex:
        print("⚠️ não consegui detectar batidas:", ex)
    return Estrutura(segs, tv, vv, beats, "áudio")

# ---------------------------------------------------------------------
# 5. MONTADOR
# ---------------------------------------------------------------------
class Montador:
    def __init__(self, videos, W, H, fps, perfil, est, fator, estr, titulo, estilo_titulo, hue_base):
        self.v, self.W, self.H, self.fps = videos, W, H, fps
        self.p, self.est, self.fator, self.estr, self.titulo = perfil, est, fator, estr, titulo
        self.escala_fx = fator / 0.65
        self.hue = est["hue"] if hue_base is None else hue_base
        self.i = 0
        self.taxa_rgb = perfil["rgb"] * est["rgb"] * fator * 0.5 / fps
        self.taxa_vhs = perfil["vhs"] * est["vhs"] * fator * 0.5 / fps
        self.rgb_left = self.vhs_left = self.rgb_int = 0
        self.q_frames = [int(round(q * fps)) for q in estr.quedas]
        self.q_ptr, self.ult_queda = 0, -99.0
        self.hist = deque(maxlen=4)
        self.fb, self.maps_k = None, {}
        self.main = self.out = self.sec = None
        self.trans_left = self.trans_total = 1
        self.ultimo = None
        self._pal_i, self._pal = -1, None
        self.tmask = TituloMask(titulo.upper(), W, H) if (titulo and estilo_titulo == "Máscara (janela)") else None
        self._novo_corte(primeiro=True)

    # ---- cortes --------------------------------------------------------
    def _duracao(self, intens):
        c_min, c_max = self.p["corte"]
        esc = 1.5 - 1.0 * intens                        # calmo = cortes longos; intenso = curtos
        c_min, c_max = c_min * esc, c_max * esc
        t, b = self.i / self.fps, self.estr.beats
        if b is not None and len(b) > 16:               # corte termina numa batida (múltiplos de 4)
            j = int(np.searchsorted(b, t))
            if j > 0 and (j >= len(b) or abs(b[j - 1] - t) <= abs(b[j] - t)):
                j -= 1
            if j < len(b) - 10:
                per = float(np.median(np.diff(b[j:j + 9])))
                lo = max(1, round(c_min / per / 4))
                hi = max(lo, round(c_max / per / 4))
                nb = 4 * random.randint(lo, hi)
                if intens > 0.85 and random.random() < 0.35:
                    nb = 2
                nb = min(nb, len(b) - 1 - j)            # nunca passar do fim da grade de batidas
                n = int(round(b[j + nb] * self.fps)) - self.i
                if n >= 3:
                    return n
        return max(4, int(random.uniform(c_min, c_max) * self.fps))

    def _abrir(self, n, evitar=None):
        opts = [x for x in self.v if x not in (self.ultimo, evitar)] or self.v
        path = random.choice(opts)
        return Clip(path, self.W, self.H, self.fps, n), path

    @staticmethod
    def _fechar(c):
        if c is not None:
            c.close()

    def _escolher_cena(self, fase):
        mult, esc = self.est["fx"], self.escala_fx
        cand = [k for k, p in POOL[fase].items() if random.random() < min(0.9, p * mult.get(k, 1.0) * esc)]
        pesados = [k for k in cand if k in ("kaleido", "neon", "wave", "fb")]
        while len(pesados) > 2:                         # no máx. 2 efeitos pesados juntos
            r = random.choice(pesados)
            pesados.remove(r)
            cand.remove(r)
        return dict(
            pb=random.random() < self.est["pb"],
            kaleido=random.choice([4, 6, 8]) if "kaleido" in cand else 0,
            kal_vel=random.uniform(3, 10) * random.choice([-1, 1]),
            neon="neon" in cand,
            gmap=random.uniform(0.65, 1.0) if "gmap" in cand else 0.0,
            fb="fb" in cand, fb_decay=random.uniform(0.86, 0.94),
            fb_s=random.uniform(1.008, 1.025), fb_rot=random.uniform(-1.0, 1.0),
            wave="wave" in cand,
            scan="scan" in cand,
            ger=random.choice(["tunel", "lasers", "grade"]) if "ger" in cand else None,
            push=random.uniform(1.0, 1.08) if random.random() < 0.7 else 1.0,
        )

    def _novo_corte(self, primeiro=False, seco=False, queda=False):
        t = self.i / self.fps
        fase, intens = self.estr.fase(t), self.estr.intens(t)
        if queda:
            fase, intens = "drop", max(intens, 0.9)
        n = self._duracao(intens)
        prox = [q for q in self.q_frames[self.q_ptr:] if q > self.i]
        if prox and self.i + n > prox[0]:               # o corte termina exatamente no próximo drop
            n = max(2, prox[0] - self.i)
        novo, path = self._abrir(n)
        if not primeiro:
            p_fade = min(0.97, max(0.05, self.p["fade"] * (1.15 - 0.6 * intens)))
            if not seco and random.random() < p_fade:  # crossfade
                self._fechar(self.out)
                self.out = self.main
                self.trans_total = self.trans_left = max(3, int(self.fps * random.uniform(0.5, 1.2)))
            else:                                       # corte seco (+ impacto de glitch)
                self._fechar(self.main)
                if queda:
                    self.rgb_left, self.rgb_int = 6, int(self.W * 0.014)
                elif random.random() < 0.4 * self.fator * self.est["rgb"]:
                    self.rgb_left, self.rgb_int = 4, int(self.W * 0.01)
        self.main, self.ultimo, self.restante, self.n_corte = novo, path, n, n
        self.cena = self._escolher_cena(fase)
        self._fechar(self.sec)
        self.sec = None
        if random.random() < min(0.95, self.p["ovl"] * self.fator * (0.6 + 0.8 * intens)):
            self.sec_dur = int(n * random.uniform(0.5, 1.0))
            self.sec, _ = self._abrir(self.sec_dur, evitar=path)
            self.sec_t = 0
            self.sec_modo = random.choice(["screen", "lighten"])

    def pular(self, n):
        self.i += n
        while self.q_ptr < len(self.q_frames) and self.q_frames[self.q_ptr] <= self.i:
            self.q_ptr += 1

    def _pal_cor(self, t):
        if self._pal_i != self.i:
            self._pal = paleta_lut((self.hue + t * self.est["hue_vel"]) % 180)
            self._pal_i = self.i
        return self._pal

    # ---- um frame ------------------------------------------------------
    def proximo(self):
        t = self.i / self.fps
        if self.q_ptr < len(self.q_frames) and self.i >= self.q_frames[self.q_ptr]:
            self.q_ptr += 1
            self.ult_queda = t
            self._novo_corte(seco=True, queda=True)
        elif self.restante <= 0:
            self._novo_corte()
        c, W, H = self.cena, self.W, self.H
        intens = self.estr.intens(t)
        nxt = (self.q_frames[self.q_ptr] / self.fps - t) if self.q_ptr < len(self.q_frames) else None
        pre = nxt is not None and 0 < nxt <= 4.0        # janela de build antes do drop

        f = self.main.read()
        self.restante -= 1
        self.hist.append(f)
        if nxt is not None and 0 < nxt <= 1.2 and len(self.hist) == 4:   # gagueira antes do drop
            f = self.hist[(self.i // 2) % 4]

        if self.trans_left > 0 and self.out is not None:                 # crossfade
            a = self.trans_left / self.trans_total
            f = cv2.addWeighted(f, 1 - a, self.out.read(), a, 0)
            self.trans_left -= 1
            if self.trans_left == 0:
                self._fechar(self.out)
                self.out = None

        if self.sec is not None:                                          # overlay de 2º vídeo
            fade = max(1, int(self.fps * 0.8))
            env = min(1.0, self.sec_t / fade, (self.sec_dur - self.sec_t) / fade)
            if env > 0:
                f = blend(f, self.sec.read(), 0.55 * env, self.sec_modo)
            self.sec_t += 1
            if self.sec_t >= self.sec_dur:
                self._fechar(self.sec)
                self.sec = None

        bt = self.estr.batida(t)
        pulso = (bt[1] * math.exp(-bt[0] * 5)) if bt else 0.0

        if c["ger"]:                                                      # camada gerada (túnel/lasers/grid)
            fade = max(1, int(self.fps * 0.7))
            env = min(1.0, (self.n_corte - self.restante) / fade, self.restante / fade)
            if env > 0:
                lut, bgr = self._pal_cor(t)
                f = blend(f, camada_gerada(c["ger"], t, W, H, bgr[2], pulso), 0.6 * env, "screen")

        # zoom: push-in lento no corte + punch no beat + aproximação antes do drop
        prog = 1 - self.restante / max(1, self.n_corte)
        s = 1 + (c["push"] - 1) * prog
        if bt:
            s *= 1 + 0.05 * self.fator * intens * bt[1] * math.exp(-bt[0] * 6)
        if pre:
            s *= 1 + 0.12 * (1 - nxt / 4.0)
        f = zoom_rot(f, s, 1.2 * math.sin(t * 0.4) * (1 - intens))

        if c["wave"]:
            f = onda(f, t, W * 0.008 * (0.6 + 0.8 * intens))
        if c["kaleido"]:
            N = c["kaleido"]
            if (N, W, H) not in self.maps_k:
                self.maps_k[(N, W, H)] = mapas_kaleido(W, H, N)
            f = kaleido(f, self.maps_k[(N, W, H)], t * c["kal_vel"])
        if pre:
            f = pixelar(f, int(2 + 22 * (1 - nxt / 4.0) ** 2))
        if c["neon"]:
            f = neon(f, self._pal_cor(t)[1][2])
        if c["gmap"] > 0:
            f = gradient_map(f, self._pal_cor(t)[0], c["gmap"])

        dq = t - self.ult_queda                                           # flash suave no drop
        flash = (1 - dq / 0.6) ** 2 if 0 <= dq < 0.6 else 0.0
        brilho = self.p["brilho"] + 18 * self.fator * pulso * (0.4 + 0.6 * intens) + FLASH_DROP * self.fator * flash
        f = grade(f, self.p["contraste"], brilho, c["pb"])

        if c["fb"]:                                                       # feedback / rastro
            if self.fb is not None:
                M = cv2.getRotationMatrix2D((W / 2, H / 2), c["fb_rot"], c["fb_s"])
                w = cv2.warpAffine(self.fb, M, (W, H), flags=cv2.INTER_LINEAR)
                f = cv2.max(f, cv2.convertScaleAbs(w, alpha=c["fb_decay"]))
            self.fb = f
        else:
            self.fb = None

        k = 0.3 + 1.4 * intens                                            # glitch em rajadas, mais forte com a energia
        if self.rgb_left <= 0 and random.random() < self.taxa_rgb * k:
            self.rgb_left = random.randint(3, 9)
            self.rgb_int = int(W * random.uniform(0.005, 0.008 + 0.012 * self.fator))
        if self.rgb_left > 0:
            f = rgb_split(f, max(1, self.rgb_int + random.randint(-2, 2)))
            self.rgb_left -= 1
        if self.vhs_left <= 0 and random.random() < self.taxa_vhs * k:
            self.vhs_left = random.randint(3, 8)
        if self.vhs_left > 0:
            f = vhs_glitch(f)
            self.vhs_left -= 1

        if c["scan"]:
            f = scanlines(f)

        if self.titulo:
            ph = t % TITULO_INTERVALO
            if ph < TITULO_VISIVEL:
                a = max(0.0, min(1.0, ph / 0.5, (TITULO_VISIVEL - ph) / 0.5))
                f = self.tmask.aplicar(f, a) if self.tmask else desenhar_titulo(f, self.titulo.upper(), a, W, H)

        self.i += 1
        return f

# ---------------------------------------------------------------------
# 6. EXPORTAÇÃO: frames vão direto pro ffmpeg
# ---------------------------------------------------------------------
def abrir_ffmpeg(saida, W, H, fps, crf, preset, log):
    cmd = ["ffmpeg", "-y", "-loglevel", "error",
           "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{W}x{H}", "-r", str(fps), "-i", "-",
           "-an",                                   # sem áudio
           "-c:v", "libx264", "-preset", preset, "-crf", str(crf),
           "-pix_fmt", "yuv420p", "-g", str(fps * 2), "-movflags", "+faststart", saida]
    return subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=log)

def renderizar(videos, W, H, fps, dur_seg, perfil, est, fator, estr, titulo, estilo_titulo,
               hue_base, crf, preset, nome_final, seed):
    if seed:
        random.seed(seed)
        np.random.seed(seed)
    total = dur_seg * fps
    bloco = BLOCO_SEG * fps
    n_blocos = math.ceil(total / bloco)
    pasta = f"/content/blocos_{nome_final[:-4]}"
    os.makedirs(pasta, exist_ok=True)
    m = Montador(videos, W, H, fps, perfil, est, fator, estr, titulo, estilo_titulo, hue_base)
    partes, t0, feitos = [], time.time(), 0

    for b in range(n_blocos):
        n = min(bloco, total - b * bloco)
        arq = f"{pasta}/bloco_{b:03d}.mp4"
        partes.append(arq)
        if os.path.exists(arq) and os.path.getsize(arq) > 0:
            print(f"⏭️ bloco {b + 1}/{n_blocos} já existe, pulando")
            m.pular(n)
            continue
        tmp = arq.replace(".mp4", ".part.mp4")
        with open(f"{pasta}/ffmpeg_{b:03d}.log", "w") as log:
            proc = abrir_ffmpeg(tmp, W, H, fps, crf, preset, log)
            try:
                for _ in tqdm(range(n), desc=f"Bloco {b + 1}/{n_blocos}", leave=False):
                    proc.stdin.write(np.ascontiguousarray(m.proximo()).tobytes())
            except BrokenPipeError:
                pass
            finally:
                try:
                    proc.stdin.close()
                except Exception:
                    pass
                proc.wait()
        if proc.returncode != 0:
            raise RuntimeError(f"ffmpeg falhou no bloco {b}. Veja {pasta}/ffmpeg_{b:03d}.log")
        os.replace(tmp, arq)
        feitos += n
        vel = feitos / (time.time() - t0)
        restante = (total - (b * bloco + n)) / max(vel, 1e-6)
        print(f"✔ bloco {b + 1}/{n_blocos} | {vel:.1f} fps de render | falta ~{restante / 60:.1f} min")

    lista = f"{pasta}/lista.txt"
    with open(lista, "w") as f:
        for p in partes:
            f.write(f"file '{p}'\n")
    final_local = f"/content/{nome_final}"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", lista,
                    "-c", "copy", "-an", "-movflags", "+faststart", final_local], check=True)
    destino = os.path.join(OUTPUT, nome_final)
    shutil.copy2(final_local, destino)
    shutil.rmtree(pasta, ignore_errors=True)         # libera disco (a cópia no Drive já foi feita)
    return destino

# ---------------------------------------------------------------------
# 7. PAINEL
# ---------------------------------------------------------------------
L = widgets.Layout(width="95%")
SEM_AUDIO = None
w_duracao = widgets.Dropdown(options=[("1 minuto (teste)", 1), ("5 minutos", 5), ("15 minutos", 15), ("30 minutos", 30),
                                      ("45 minutos", 45), ("60 minutos", 60), ("90 minutos", 90), ("120 minutos", 120)],
                             value=1, description="Duração:", layout=L)
w_genero = widgets.Dropdown(options=list(PERFIS_GENERO), value="Melodic Techno", description="Gênero:", layout=L)
w_estetica = widgets.Dropdown(options=list(ESTETICAS), value="Dark / Industrial", description="Estética:", layout=L)
w_paleta = widgets.Dropdown(options=list(PALETAS), value="Auto (pela estética)", description="Paleta:", layout=L)
w_res = widgets.Dropdown(options=list(RESOLUCOES), value=list(RESOLUCOES)[0], description="Resolução:", layout=L)
w_fps = widgets.Dropdown(options=[24, 25, 30, 60], value=30, description="FPS:", layout=L)
w_fx = widgets.IntSlider(value=65, min=0, max=100, description="Efeitos:", layout=L)
w_qual = widgets.Dropdown(options=[("Boa (crf 20, rápido)", (20, "veryfast")), ("Alta (crf 17, mais lento)", (17, "medium"))],
                          value=(20, "veryfast"), description="Qualidade:", layout=L)

w_sec_estr = widgets.HTML("<b>Estrutura do set (build / drop / breakdown)</b><br>"
                          "<small>Prioridade: 1) áudio analisado, 2) timeline digitada, 3) ciclo automático.</small>")
w_audio = widgets.Dropdown(options=[("— sem áudio (usar timeline / ciclo) —", SEM_AUDIO)] +
                                   [(os.path.basename(a), a) for a in audio_files],
                           value=SEM_AUDIO, description="Áudio:", layout=L)
w_timeline = widgets.Textarea(placeholder="0:00-4:00 breakdown; 4:00-6:00 build; 6:00-14:00 drop",
                              description="Timeline:", layout=widgets.Layout(width="95%", height="60px"))
w_ciclo = widgets.IntText(value=8, description="Ciclo (min):", layout=L)
w_bpm = widgets.IntText(value=0, description="BPM:", layout=L)
w_info = widgets.HTML("<small>Timeline: 'mm:ss' ou só minutos (ex.: 0-4 breakdown). BPM só vale sem áudio "
                      "(0 = cortes em tempo livre). O áudio é usado apenas para análise; o vídeo sai mudo.</small>")

w_titulo = widgets.Text(placeholder="Ex: ALDECAYDO / DEEP GLITCH", description="Título:", layout=L)
w_estilo_tit = widgets.Dropdown(options=["Máscara (janela)", "Rodapé"], value="Máscara (janela)",
                                description="Estilo:", layout=L)
w_seed = widgets.IntText(value=0, description="Seed:", layout=L)
w_botao = widgets.Button(description="GERAR VISUAL MIX", button_style="success", icon="play",
                         layout=widgets.Layout(width="95%", height="45px"))
w_out = widgets.Output()

def ao_clicar(_):
    with w_out:
        clear_output()
        vids = validar_videos(video_files)
        if not vids:
            print("❌ Nenhum vídeo válido em video_files. Confira VIDEO_DIR.")
            return
        W, H = RESOLUCOES[w_res.value]
        dur_min, bpm, fps = w_duracao.value, max(0, w_bpm.value), w_fps.value
        dur_seg = dur_min * 60
        crf, preset = w_qual.value
        beats_bpm = np.arange(0, dur_seg + 4, 60.0 / bpm) if bpm > 0 else None

        estr = None
        if w_audio.value:
            try:
                estr = estrutura_audio(w_audio.value, dur_seg)
                if estr.beats is None:
                    estr.beats = beats_bpm
            except ImportError:
                print("⚠️ librosa não instalada: rode `!pip install librosa` e tente de novo. Usando timeline/ciclo.")
            except Exception as ex:
                print(f"⚠️ falha na análise do áudio ({ex}). Usando timeline/ciclo.")
        if estr is None and w_timeline.value.strip():
            estr = estrutura_timeline(w_timeline.value, beats_bpm)
            if estr is None:
                print("⚠️ não entendi a timeline. Exemplo: 0:00-4:00 breakdown; 4:00-6:00 build; 6:00-14:00 drop")
        if estr is None:
            estr = estrutura_auto(dur_seg, w_ciclo.value, beats_bpm)

        cfg = "|".join(map(str, [w_genero.value, w_estetica.value, w_paleta.value, w_fx.value, w_audio.value,
                                 w_timeline.value, w_ciclo.value, bpm, w_titulo.value, w_estilo_tit.value,
                                 w_seed.value, dur_min, W, H, fps, crf]))
        tag = f"{zlib.crc32(cfg.encode()):08x}"[:6]
        nome = f"VISUAL_{w_genero.value.replace(' ', '_')}_{dur_min}min_{W}x{H}_{fps}fps_{tag}_TELAO_MUDO.mp4"
        print(f"🎛️ {w_genero.value} | {w_estetica.value} | {dur_min} min | {W}x{H} @ {fps}fps | {len(vids)} clipes")
        print(estr.resumo())
        print("🥁 Batidas: " + ("grade detectada/BPM definido (cortes em compassos)" if estr.beats is not None
                               else "sem grade (tempos livres)"))
        w_botao.disabled = True
        try:
            destino = renderizar(vids, W, H, fps, dur_seg, PERFIS_GENERO[w_genero.value], ESTETICAS[w_estetica.value],
                                 w_fx.value / 100.0, estr, w_titulo.value.strip(), w_estilo_tit.value,
                                 PALETAS[w_paleta.value], crf, preset, nome, w_seed.value)
            print(f"\n✅ PRONTO (sem áudio): {destino}")
        finally:
            w_botao.disabled = False

w_botao.on_click(ao_clicar)
display(widgets.VBox([widgets.HTML("<h2>🎬 VJ MIX GENERATOR v3</h2>"),
                      w_duracao, w_genero, w_estetica, w_paleta, w_res, w_fps, w_fx, w_qual,
                      w_sec_estr, w_audio, w_timeline, w_ciclo, w_bpm, w_info,
                      w_titulo, w_estilo_tit, w_seed, w_botao, w_out]))